# Reproduction notebook: 39_long_horizon_predictive_relevance_matched_protocol_v2_autoclone

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:
from pathlib import Path
import os
import re
import json
import copy
import shutil
import subprocess
import sys
import time
import zipfile
import pandas as pd
import numpy as np

HORIZONS = [96, 192, 336, 720]

# ---------------------------------------------------------------------
# User-overridable configuration
# ---------------------------------------------------------------------
REPO_ROOT_OVERRIDE = os.environ.get("WHM_REPO_ROOT", "").strip()
DATA_ROOT_OVERRIDE = os.environ.get("WHM_DATA_ROOT", "").strip()

DEFAULT_WORK_ROOT = Path(
    os.environ.get(
        "WHM_LONGH_WORK_ROOT",
        "/data/dataset/strong_forecaster/predictive_relevance_long_horizon",
    )
)

WORK_ROOT = DEFAULT_WORK_ROOT.resolve()
PATCHED_DIR = WORK_ROOT / "patched_notebooks"
EXECUTED_DIR = WORK_ROOT / "executed_notebooks"
BUNDLE_DIR = WORK_ROOT / "bundle"

for p in [WORK_ROOT, PATCHED_DIR, EXECUTED_DIR, BUNDLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Target horizons:", HORIZONS)
print("Work root:", WORK_ROOT)


In [ ]:
def looks_like_repo(p):
    p = Path(p)
    return (
        (p / "README.md").is_file()
        and (p / "experiments/confirmatory/confirmatory_benchmark.ipynb").is_file()
        and (p / "experiments/mechanism/00_cross_domain_base.ipynb").is_file()
        and (p / "experiments/mechanism/01_etth1_weather_relevance.ipynb").is_file()
        and (p / "experiments/mechanism/02_candidate_prior.ipynb").is_file()
    )


def locate_repo():
    candidates = []

    if REPO_ROOT_OVERRIDE:
        candidates.append(Path(REPO_ROOT_OVERRIDE))

    candidates += [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/data/code/which-histories-matter"),
        Path("/data/which-histories-matter"),
        Path("/data/code/Which-Histories-Matter"),
        Path("/data/pcw_workspace/which-histories-matter"),
    ]

    seen = set()

    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue

        if str(p) in seen:
            continue

        seen.add(str(p))

        if looks_like_repo(p):
            return p

    # Keep the recursive search intentionally small.
    for root in [
        Path("/data/code"),
        Path("/data/pcw_workspace"),
    ]:
        if not root.exists():
            continue

        try:
            hits = list(
                root.glob(
                    "*/experiments/confirmatory/confirmatory_benchmark.ipynb"
                )
            )
        except Exception:
            hits = []

        for candidate in hits:
            repo = candidate.parents[2]

            if looks_like_repo(repo):
                return repo.resolve()

    return None


def _can_write_parent(path):
    path = Path(path)
    parent = path.parent

    try:
        parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        probe = (
            parent
            / ".whm_write_probe"
        )

        probe.write_text(
            "ok",
            encoding="utf-8",
        )

        probe.unlink()

        return True

    except Exception:
        return False


def clone_public_repo():
    auto_clone = os.environ.get(
        "WHM_AUTO_CLONE",
        "1",
    ).strip().lower()

    if auto_clone in {
        "0",
        "false",
        "no",
    }:
        return None

    clone_url = (
        "https://github.com/"
        "anonymous_repository/"
        "which-histories-matter.git"
    )

    destinations = []

    if REPO_ROOT_OVERRIDE:
        destinations.append(
            Path(REPO_ROOT_OVERRIDE)
        )

    destinations += [
        Path(
            "/data/code/"
            "which-histories-matter"
        ),
        Path(
            "/data/"
            "which-histories-matter"
        ),
        WORK_ROOT
        / "which-histories-matter",
    ]

    for dst in destinations:
        dst = dst.expanduser()

        if looks_like_repo(dst):
            return dst.resolve()

        if dst.exists():
            # Do not delete or overwrite an unrelated directory.
            if any(dst.iterdir()):
                print(
                    "Skip non-empty clone destination:",
                    dst,
                )
                continue

        if not _can_write_parent(dst):
            print(
                "Clone destination is not writable:",
                dst,
            )
            continue

        print(
            "Local repository not found."
        )

        print(
            "Cloning public repository to:",
            dst,
        )

        cmd = [
            "git",
            "clone",
            "--depth",
            "1",
            clone_url,
            str(dst),
        ]

        proc = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        print(
            proc.stdout[-8000:]
        )

        if (
            proc.returncode == 0
            and looks_like_repo(dst)
        ):
            print(
                "Clone completed successfully."
            )

            return dst.resolve()

        print(
            "Clone attempt failed for:",
            dst,
        )

    return None


REPO_ROOT = locate_repo()

if REPO_ROOT is None:
    REPO_ROOT = clone_public_repo()

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find or automatically clone "
        "anonymous_repository/which-histories-matter.\n\n"
        "Check that git and outbound HTTPS access are available, or run:\n"
        "  git clone https://anonymous_repository/"
        "which-histories-matter.git "
        "/data/code/which-histories-matter\n\n"
        "Then rerun this notebook, or set WHM_REPO_ROOT."
    )

print(
    "Repository root:",
    REPO_ROOT,
)

try:
    git_head = subprocess.check_output(
        [
            "git",
            "rev-parse",
            "HEAD",
        ],
        cwd=REPO_ROOT,
        text=True,
    ).strip()

except Exception:
    git_head = "unknown"

print(
    "Git HEAD:",
    git_head,
)

# Show the exact public notebooks required by the controller.
_required_public_notebooks = [
    REPO_ROOT / "experiments/mechanism/00_cross_domain_base.ipynb",
    REPO_ROOT / "experiments/mechanism/01_etth1_weather_relevance.ipynb",
    REPO_ROOT / "experiments/mechanism/02_candidate_prior.ipynb",
    REPO_ROOT / "experiments/confirmatory/confirmatory_benchmark.ipynb",
]

for p in _required_public_notebooks:
    print(
        "FOUND:",
        p,
    )

    if not p.is_file():
        raise FileNotFoundError(
            p
        )

print(
    "PASS: public predictive-relevance notebooks are available."
)


In [ ]:
REQUIRED_DATA = {
    "ETTh1": Path("ETT-small/ETTh1.csv"),
    "Weather": Path("weather/weather.csv"),
    "Electricity": Path("electricity/electricity.csv"),
    "Traffic": Path("traffic/traffic.csv"),
    "Exchange": Path("exchange_rate/exchange_rate.csv"),
    "Solar": Path("Solar/solar_AL.txt"),
}


def data_coverage(root):
    root = Path(root)
    return {
        name: (root / rel).is_file()
        for name, rel in REQUIRED_DATA.items()
    }


def locate_data_root():
    candidates = []

    if DATA_ROOT_OVERRIDE:
        candidates.append(Path(DATA_ROOT_OVERRIDE))

    candidates += [
        Path("/data/Time-Series-Library_v2/dataset"),
        Path("/data/Time-Series-Library/dataset"),
        Path("/data/pcw_workspace/Time-Series-Library/dataset"),
        Path("/data/pcw_workspace/Time-Series-Library_v2/dataset"),
        Path("/data/dataset"),
        REPO_ROOT / "data",
    ]

    best = None
    best_n = -1

    for root in candidates:
        if not root.exists():
            continue

        cov = data_coverage(root)
        n = sum(cov.values())

        if n > best_n:
            best = root.resolve()
            best_n = n

        if n == len(REQUIRED_DATA):
            return root.resolve()

    if best is not None:
        print("Best partial dataset root:", best)
        print(data_coverage(best))

    return None


DATA_ROOT = locate_data_root()

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find a dataset root containing all six benchmark files. "
        "Set WHM_DATA_ROOT to the Time-Series-Library dataset directory."
    )

coverage = data_coverage(DATA_ROOT)

display(
    pd.DataFrame(
        [
            {
                "Dataset": name,
                "Path": str(DATA_ROOT / rel),
                "Exists": coverage[name],
            }
            for name, rel in REQUIRED_DATA.items()
        ]
    )
)

if not all(coverage.values()):
    missing = [k for k, v in coverage.items() if not v]
    raise FileNotFoundError(
        f"Dataset root {DATA_ROOT} is missing: {missing}"
    )

print("PASS: all six datasets found.")
print("Data root:", DATA_ROOT)


In [ ]:
SRC = {
    "base": REPO_ROOT / "experiments/mechanism/00_cross_domain_base.ipynb",
    "mechanism": REPO_ROOT / "experiments/mechanism/01_etth1_weather_relevance.ipynb",
    "prior": REPO_ROOT / "experiments/mechanism/02_candidate_prior.ipynb",
    "confirmatory": REPO_ROOT / "experiments/confirmatory/confirmatory_benchmark.ipynb",
}

for k, p in SRC.items():
    if not p.is_file():
        raise FileNotFoundError(p)

PATCHED = {
    "base": PATCHED_DIR / "00_cross_domain_base_longH.ipynb",
    "mechanism": PATCHED_DIR / "01_etth1_weather_relevance_longH.ipynb",
    "prior": PATCHED_DIR / "02_candidate_prior_longH.ipynb",
    "confirmatory": PATCHED_DIR / "confirmatory_benchmark_longH.ipynb",
}


def cell_text(cell):
    return "".join(cell.get("source", []))


def set_cell_text(cell, text):
    cell["source"] = text.splitlines(keepends=True)


def replace_horizons(text):
    pattern = re.compile(
        r"HORIZONS\s*=\s*\[\s*24,\s*48,\s*96,\s*\]",
        flags=re.S,
    )

    repl = (
        "HORIZONS = [\n"
        "    96,\n"
        "    192,\n"
        "    336,\n"
        "    720,\n"
        "]"
    )

    text, n = pattern.subn(repl, text)

    if n == 0:
        raise RuntimeError("Could not locate HORIZONS=[24,48,96] block.")

    return text


def replace_dataset_names_base(text):
    pattern = re.compile(
        r'DATASET_NAMES\s*=\s*\[\s*"ETTh1",\s*"Weather",\s*"Electricity",\s*"Traffic",\s*\]',
        flags=re.S,
    )

    repl = (
        'DATASET_NAMES = [\n'
        '    "ETTh1",\n'
        '    "Weather",\n'
        ']'
    )

    text, n = pattern.subn(repl, text)

    if n == 0:
        raise RuntimeError("Could not restrict base DATASET_NAMES to ETTh1/Weather.")

    return text


def patch_root_bootstrap(text):
    # Preserve the public notebook's logic but inject exact roots before it executes.
    if "_env_repo = os.environ.get(\"WHM_REPO_ROOT\")" in text:
        text = text.replace(
            '_env_repo = os.environ.get("WHM_REPO_ROOT")',
            f'_env_repo = {str(REPO_ROOT)!r}',
        )

    if 'REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT"' in text:
        # The original expression will now read the environment variable set by execute_notebook.
        pass

    return text


def patch_notebook(kind, src, dst):
    nb = json.loads(src.read_text(encoding="utf-8"))

    horizon_replaced = False
    dataset_replaced = False

    for cell in nb["cells"]:
        text = cell_text(cell)

        # Update explanatory markdown too.
        text = (
            text
            .replace(r"H\in\{24,48,96\}", r"H\in\{96,192,336,720\}")
            .replace(r"H\in\{24,48,96\\}", r"H\in\{96,192,336,720\\}")
            .replace("24,48,96", "96,192,336,720")
        )

        if cell.get("cell_type") == "code":
            if "HORIZONS" in text and all(token in text for token in ["24", "48", "96"]):
                try:
                    text = replace_horizons(text)
                    horizon_replaced = True
                except RuntimeError:
                    pass

            text = patch_root_bootstrap(text)

            if kind == "base" and "DATASET_NAMES" in text and '"Electricity"' in text and '"Traffic"' in text:
                try:
                    text = replace_dataset_names_base(text)
                    dataset_replaced = True
                except RuntimeError:
                    pass

        set_cell_text(cell, text)

    if not horizon_replaced:
        raise RuntimeError(f"{kind}: HORIZONS block was not patched.")

    if kind == "base" and not dataset_replaced:
        raise RuntimeError("base: DATASET_NAMES block was not patched.")

    # Directory provenance patches.
    for cell in nb["cells"]:
        text = cell_text(cell)

        if kind == "base":
            text = text.replace(
                'REPO_WORK_ROOT / "cross_domain_clean"',
                'REPO_WORK_ROOT / "long_horizon_cross_domain_base"',
            )

        elif kind == "mechanism":
            text = text.replace(
                'REPO_WORK_ROOT / "cross_domain_clean"',
                'REPO_WORK_ROOT / "long_horizon_cross_domain_base"',
            )
            text = text.replace(
                'REPO_WORK_ROOT / "semantic_compatibility"',
                'REPO_WORK_ROOT / "long_horizon_semantic_compatibility"',
            )

        elif kind == "prior":
            text = text.replace(
                'REPO_WORK_ROOT / "cross_domain_clean"',
                'REPO_WORK_ROOT / "long_horizon_cross_domain_base"',
            )
            text = text.replace(
                'REPO_WORK_ROOT / "semantic_compatibility"',
                'REPO_WORK_ROOT / "long_horizon_semantic_compatibility"',
            )
            text = text.replace(
                'REPO_WORK_ROOT / "candidate_prior_deconfounding"',
                'REPO_WORK_ROOT / "long_horizon_candidate_prior_deconfounding"',
            )

        elif kind == "confirmatory":
            text = text.replace(
                'REPO_WORK_ROOT / "final_confirmatory"',
                'REPO_WORK_ROOT / "long_horizon_confirmatory"',
            )

        set_cell_text(cell, text)

    # Clear any old execution state.
    for cell in nb["cells"]:
        if cell.get("cell_type") == "code":
            cell["execution_count"] = None
            cell["outputs"] = []

    dst.write_text(
        json.dumps(nb, ensure_ascii=False, indent=1),
        encoding="utf-8",
    )

    return nb


for kind in ["base", "mechanism", "prior", "confirmatory"]:
    patch_notebook(kind, SRC[kind], PATCHED[kind])
    print(f"PATCHED {kind:12s} -> {PATCHED[kind]}")

print("PASS: four long-horizon notebooks prepared.")


In [ ]:
audit_rows = []

expected_tokens = [
    "TOP_M = 100",
    "TOP_K = 10",
    "TAU_Y = 0.50",
    "TRAIN_BATCH = 128",
    "MAX_EPOCHS = 30",
    "PATIENCE = 6",
    "LR = 1e-3",
    "WEIGHT_DECAY = 1e-4",
]

for kind, path in PATCHED.items():
    nb = json.loads(path.read_text(encoding="utf-8"))
    text = "\n".join(cell_text(c) for c in nb["cells"])

    row = {
        "Notebook": kind,
        "HasH96": "96" in text,
        "HasH192": "192" in text,
        "HasH336": "336" in text,
        "HasH720": "720" in text,
        "NoOldHorizonBlock": not bool(
            re.search(
                r"HORIZONS\s*=\s*\[\s*24,\s*48,\s*96,\s*\]",
                text,
                flags=re.S,
            )
        ),
    }

    for token in expected_tokens:
        row[token] = token in text

    audit_rows.append(row)

audit = pd.DataFrame(audit_rows)
display(audit)

if not bool(audit.drop(columns=["Notebook"]).all().all()):
    raise RuntimeError("Patch audit failed.")

print("PASS: frozen recipe retained and horizon patch verified.")


In [ ]:
EXECUTION_ORDER = [
    "base",
    "mechanism",
    "prior",
    "confirmatory",
]

ENV = os.environ.copy()
ENV["WHM_REPO_ROOT"] = str(REPO_ROOT)
ENV["WHM_DATA_ROOT"] = str(DATA_ROOT)
ENV["WHM_WORK_ROOT"] = str(WORK_ROOT)

# Can be set before running this notebook if a different kernel is needed.
KERNEL_NAME = os.environ.get("WHM_KERNEL_NAME", "python3")
CELL_TIMEOUT = int(os.environ.get("WHM_CELL_TIMEOUT", "7200"))

print("Kernel:", KERNEL_NAME)
print("Per-cell timeout:", CELL_TIMEOUT, "seconds")
print("Execution work root:", WORK_ROOT)


In [ ]:
def execute_notebook(kind):
    src = PATCHED[kind]
    dst = EXECUTED_DIR / src.name.replace(".ipynb", "_executed.ipynb")

    cmd = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        str(src),
        "--output",
        str(dst),
        "--ExecutePreprocessor.kernel_name=" + KERNEL_NAME,
        "--ExecutePreprocessor.timeout=" + str(CELL_TIMEOUT),
    ]

    print("=" * 110)
    print("EXECUTING:", kind)
    print("SOURCE   :", src)
    print("OUTPUT   :", dst)
    print("=" * 110)

    t0 = time.time()

    proc = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    elapsed = (time.time() - t0) / 60.0

    print(proc.stdout[-12000:])
    print(f"Elapsed: {elapsed:.1f} min")

    if proc.returncode != 0:
        raise RuntimeError(
            f"{kind} failed with exit code {proc.returncode}. "
            f"Inspect {dst if dst.exists() else src} and the log above."
        )

    return dst, elapsed


RUN = {
    "base": True,
    "mechanism": True,
    "prior": True,
    "confirmatory": True,
}

execution_log = []

for kind in EXECUTION_ORDER:
    if not RUN[kind]:
        print("SKIP:", kind)
        continue

    dst, elapsed = execute_notebook(kind)

    execution_log.append({
        "Notebook": kind,
        "ExecutedPath": str(dst),
        "Minutes": elapsed,
    })

execution_log = pd.DataFrame(execution_log)
display(execution_log)

execution_log.to_csv(
    WORK_ROOT / "execution_log.csv",
    index=False,
)


In [ ]:
def output_text(output):
    parts = []

    if "text" in output:
        val = output["text"]
        parts.append("".join(val) if isinstance(val, list) else str(val))

    data = output.get("data", {})

    for key in ["text/plain", "text/markdown"]:
        if key in data:
            val = data[key]
            parts.append("".join(val) if isinstance(val, list) else str(val))

    return "\n".join(parts)


def notebook_output_tail(path, max_chars=16000):
    nb = json.loads(Path(path).read_text(encoding="utf-8"))
    chunks = []

    for cell in nb["cells"]:
        for out in cell.get("outputs", []):
            txt = output_text(out).strip()
            if txt:
                chunks.append(txt)

    joined = "\n\n".join(chunks)

    return joined[-max_chars:]


for _, row in execution_log.iterrows():
    print("\n" + "#" * 120)
    print(row["Notebook"].upper())
    print("#" * 120)
    print(
        notebook_output_tail(
            row["ExecutedPath"],
            max_chars=20000,
        )
    )


In [ ]:
summary_files = []

result_dirs = [
    WORK_ROOT / "long_horizon_cross_domain_base",
    WORK_ROOT / "long_horizon_semantic_compatibility",
    WORK_ROOT / "long_horizon_candidate_prior_deconfounding",
    WORK_ROOT / "long_horizon_confirmatory",
]

for root in result_dirs:
    if not root.exists():
        continue

    for pattern in ["*.csv", "*.parquet", "*.json", "*.tex"]:
        for p in root.rglob(pattern):
            # Avoid huge cache metadata if any.
            if p.stat().st_size <= 20 * 1024 * 1024:
                summary_files.append(p)

inventory = pd.DataFrame({
    "Path": [str(p) for p in sorted(set(summary_files))],
    "SizeKB": [round(p.stat().st_size / 1024, 1) for p in sorted(set(summary_files))],
})

display(inventory)

inventory.to_csv(
    WORK_ROOT / "result_file_inventory.csv",
    index=False,
)

manifest = {
    "RepositoryRoot": str(REPO_ROOT),
    "GitHEAD": git_head,
    "DataRoot": str(DATA_ROOT),
    "WorkRoot": str(WORK_ROOT),
    "Horizons": HORIZONS,
    "PatchedNotebooks": {k: str(v) for k, v in PATCHED.items()},
    "ExecutedNotebooks": execution_log.to_dict(orient="records"),
}

manifest_path = WORK_ROOT / "long_horizon_relevance_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

bundle_path = BUNDLE_DIR / "experiment39_long_horizon_relevance_results.zip"

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(manifest_path, manifest_path.name)
    z.write(WORK_ROOT / "execution_log.csv", "execution_log.csv")
    z.write(WORK_ROOT / "result_file_inventory.csv", "result_file_inventory.csv")

    for p in EXECUTED_DIR.glob("*_executed.ipynb"):
        z.write(p, f"executed_notebooks/{p.name}")

    for p in sorted(set(summary_files)):
        try:
            rel = p.relative_to(WORK_ROOT)
        except Exception:
            rel = Path(p.name)

        z.write(p, f"results/{rel}")

print("Result bundle:", bundle_path)
print("Upload this ZIP or the executed controller notebook for analysis.")


In [ ]:
print("=" * 118)
print("EXPERIMENT 39 — LONG-HORIZON PREDICTIVE RELEVANCE")
print("=" * 118)
print("Datasets : ETTh1, Weather, Electricity, Traffic, Exchange, Solar")
print("Horizons :", HORIZONS)
print("Pattern candidate M :", 100)
print("Retrieved K :", 10)
print("Target temperature :", 0.50)
print("Protocol source : exact public which-histories-matter reproducibility notebooks")
print("Git HEAD :", git_head)
print()
print("Expected next step:")
print("  24 dataset-horizon relevance conditions")
print("    ×")
print("  4 downstream forecasting backbones")
print("  -> full same-horizon relevance-to-utility bridge analysis")
